# 크기 증강과 CutMix를 함께 바꾸면 어떤 프레임이 달라질까?

같은 사람별 분할과 초기 가중치에서 학습한 네 모델을 비교합니다. 원본 687장 중 동작 중 수동 박스가 있는 270장을 채점합니다. a03의 실제 가림 2–14번과 b02의 32–35번을 따로 봅니다.

In [ ]:
from pathlib import Path
import sys
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "wlcv/config.py").is_file())
sys.path.insert(0,str(root))

from wlcv.evaluation.plate_scale_cutmix import evaluate_new_runs, summarize, REVIEW

rescore = False
if rescore:
    evaluate_new_runs()
scores, sizes = summarize()
display(scores[scores.method.eq("Past link")])

## b02의 박스 크기와 위치 확인

면적비 1은 수동 박스와 면적이 같다는 뜻입니다. 더 작으면 원판 일부만 잡았을 수 있습니다. 위치까지 맞는지는 같은 프레임의 IoU로 확인합니다. 기준은 IoU 0.50입니다. box_dx_px와 box_dy_px는 수동 박스 중심에서 예측 박스 중심까지의 차이입니다. 원본 화면의 오른쪽과 아래쪽을 양수로 셉니다.

In [ ]:
from matplotlib import pyplot as plt
import pandas as pd

b02 = sizes[sizes.method.eq("Past link")]
display(b02[["scale","cutmix","frame","width_ratio","height_ratio","area_ratio","box_dx_px","box_dy_px","iou","match"]])
fig, axes = plt.subplots(1,2,figsize=(10,4),layout="constrained")
for (scale, cutmix), group in b02.groupby(["scale","cutmix"]):
    label = f"scale {scale}, CutMix {cutmix}"
    for axis, metric in zip(axes, ["area_ratio","iou"]):
        axis.plot(group.frame, group[metric], marker="o", label=label)
for axis, title, threshold in zip(axes, ["Predicted / manual area","Box IoU"], [1.0,0.5]):
    axis.axhline(threshold,color="grey",linestyle=":")
    axis.set(title=title, xlabel="b02 original frame", xticks=[32,33,34,35])
axes[0].legend(fontsize=8)
fig.savefig(REVIEW / "b02_size_iou.png",dpi=150)
plt.show()

## 실제 가림과 동작 전체를 함께 확인

표의 복구·손실은 CutMix 확률이 같은 두 모델에서 크기 증강을 0.50에서 0.25로 바꾼 결과입니다.

In [ ]:
for probability in (0.0,0.3):
    changes = pd.read_csv(REVIEW / f"scale_change_cutmix_{probability}.csv")
    print(f"CutMix {probability}")
    display(changes[(changes["방법"]=="Past link") &
                    changes["평가 구간"].isin(["a03 2~14번","동작 중 전체"])])

## 원본에서 수동 박스와 선택된 예측 겹쳐 보기

초록 실선은 수동 원판 박스, 주황 점선은 시간 연결까지 적용한 모델 박스입니다. 같은 프레임의 네 그림은 같은 위치를 확대합니다.

In [ ]:
from wlcv.evaluation.plate_scale_cutmix import plot_boxes

for video, frame in [("a03_sn_f_01", 14), ("b02_sn_h_02", 33)]:
    fig = plot_boxes(video, frame)
    fig.savefig(REVIEW / f"{video}_{frame}_boxes.png", dpi=130)
    plt.show()

## 학습에서 본 빈 배경 확인

새 두 모델을 b05·b06 원본 영상에도 적용했습니다. 오탐이 나온 총 프레임과 끊기지 않고 이어진 최장 길이를 셉니다. 두 영상은 학습 선수의 배경이므로 독립 검증과 별도로 봅니다.

In [ ]:
from wlcv.evaluation.plate_scale_cutmix import check_backgrounds

if rescore:
    check_backgrounds()
backgrounds = pd.read_csv(REVIEW / "training_backgrounds.csv")
display(backgrounds[backgrounds.method.eq("Past link")])